### 03.03. Setting up Indexes

In [ ]:
# Configure the Azure OpenAI connection used by both the chat model and the embedding model.
# We keep credentials in environment variables instead of hardcoding them so the notebook is
# easier to reuse and safer to share in a course environment.
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings

import os
from dotenv import load_dotenv

# Load API settings from a local .env file. This keeps secrets out of source control and makes
# the setup clearer for students who are following the exercise step by step.
load_dotenv()
api_key = os.getenv("AZURE_OPENAI_API_KEY")
azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")

# Create the chat model that will decide which product the question belongs to and answer it.
llm = AzureChatOpenAI(
    azure_deployment="gpt-5.4-mini",
    api_key=api_key,
    azure_endpoint=azure_endpoint,
    api_version=api_version,
)

# Create the embedding model used for retrieval-augmented generation (RAG).
# Embeddings convert text into vectors, which lets us compare a user question with chunks from
# the product specification documents and find the most relevant content.
embed_model = AzureOpenAIEmbeddings(
    azure_deployment="text-embedding-ada-002",
    api_key=api_key,
    azure_endpoint=azure_endpoint,
    api_version=api_version,
)

In [ ]:
# Build a separate vector index for each product specification document.
# This is the core idea behind a router: we keep product-specific context isolated so retrieval is
# more precise and the answer stays grounded in the correct source document.
from langchain_pymupdf4llm import PyMuPDF4LLMLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore

# Step 1: Prepare a shared splitter for both PDFs.
# Why: we want a consistent chunking strategy for every product manual so retrieval behaves the same
# way across both knowledge bases.
splitter = RecursiveCharacterTextSplitter(chunk_size=1024)

# -------------------------------------------------------------------
# Set up the AeroFlow index
# -------------------------------------------------------------------
# Step 1: Load the AeroFlow specification PDF page by page.
# Why: this preserves the original document structure and lets us process the material in sections
# instead of treating the whole file as a single long block.
aeroflow_documents = PyMuPDF4LLMLoader("AeroFlow_Specification_Document.pdf", mode="page").load()

# Step 2: Split the document into smaller chunks.
# Why: a product spec can be long and dense. Chunking makes it easier for similarity search to find
# the most relevant passages when a user asks a targeted question.
aeroflow_chunks = splitter.split_documents(aeroflow_documents)

# Step 3: Create an in-memory vector store and embed each chunk.
# Why: the embedding model converts each chunk into a vector so the system can compare the question
# with product content semantically instead of matching only exact words.
aeroflow_vectorstore = InMemoryVectorStore(embed_model)
aeroflow_vectorstore.add_documents(aeroflow_chunks)

# Step 4: Build the retriever for AeroFlow.
# Why: the retriever is the interface used during routing. When the router selects AeroFlow, this
# retriever fetches the most relevant product passages and passes them into the answer step.
aeroflow_retriever = aeroflow_vectorstore.as_retriever()

# -------------------------------------------------------------------
# Set up the EcoSprint index
# -------------------------------------------------------------------
# We repeat the same pattern for the second product so the router can choose between specialized
# knowledge sources instead of mixing all product details into one retrieval step.
# Step 1: Load the EcoSprint specification PDF page by page.
ecosprint_documents = PyMuPDF4LLMLoader("EcoSprint_Specification_Document.pdf", mode="page").load()

# Step 2: Split the EcoSprint document into smaller chunks.
ecosprint_chunks = splitter.split_documents(ecosprint_documents)

# Step 3: Store the embedded EcoSprint chunks in memory.
ecosprint_vectorstore = InMemoryVectorStore(embed_model)
ecosprint_vectorstore.add_documents(ecosprint_chunks)

# Step 4: Create a retriever for EcoSprint.
ecosprint_retriever = ecosprint_vectorstore.as_retriever()

### 03.04. Setup the Agentic Router

In [ ]:
from typing import Literal, TypedDict
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END

# Define a structured output schema for the router.
# The LLM must return a known product name instead of free-form text. This reduces ambiguity and
# keeps the routing logic deterministic when a question clearly belongs to one product.
class RouteChoice(BaseModel):
    product: Literal["aeroflow", "ecosprint"] = Field(
        description="Which product's specifications the question is about"
    )

# Use the LLM in structured-output mode so it returns a valid RouteChoice object.
router_llm = llm.with_structured_output(RouteChoice)

class RouterState(TypedDict):
    question: str
    product: str
    answer: str

# Route the incoming question to the correct product specialist.
# We ask the model to choose a product first because retrieval is more reliable when we search the
# correct document set rather than mixing both product knowledge bases together.
def route_question(state: RouterState) -> dict:
    choice = router_llm.invoke(
        "Aeroflow specifications: Contains information about Aeroflow : Design, features, technology, maintenance, warranty\n"
        "EcoSprint specifications: Contains information about EcoSprint : Design, features, technology, maintenance, warranty\n\n"
        f"Question: {state['question']}\n"
        "Which product's specifications should be used to answer this question?"
    )
    print(f"Selecting query engine: {choice.product}")
    return {"product": choice.product}

# Decide which branch of the graph to follow based on the selected product.
def select_route(state: RouterState) -> str:
    return state["product"]

# Each answer node retrieves only the relevant product documentation and then asks the LLM to use
# that context. This is the classic retrieval-augmented pattern: grounding the answer in source
# material instead of relying only on the model's general knowledge.
def answer_aeroflow(state: RouterState) -> dict:
    docs = aeroflow_retriever.invoke(state["question"])
    context = "\n\n".join(d.page_content for d in docs)
    response = llm.invoke(
        f"Context:\n{context}\n\nQuestion: {state['question']}\nAnswer using only the context above."
    )
    return {"answer": response.content}


def answer_ecosprint(state: RouterState) -> dict:
    docs = ecosprint_retriever.invoke(state["question"])
    context = "\n\n".join(d.page_content for d in docs)
    response = llm.invoke(
        f"Context:\n{context}\n\nQuestion: {state['question']}\nAnswer using only the context above."
    )
    return {"answer": response.content}

# Build the router graph.
# A graph makes the workflow explicit: route first, then answer using the selected product's
# retrieval pipeline. This keeps the agent's behavior easy to reason about and easy to teach.
graph = StateGraph(RouterState)
graph.add_node("route_question", route_question)
graph.add_node("answer_aeroflow", answer_aeroflow)
graph.add_node("answer_ecosprint", answer_ecosprint)

graph.add_edge(START, "route_question")
graph.add_conditional_edges(
    "route_question",
    select_route,
    {"aeroflow": "answer_aeroflow", "ecosprint": "answer_ecosprint"},
)
graph.add_edge("answer_aeroflow", END)
graph.add_edge("answer_ecosprint", END)

router_agent = graph.compile()

### 03.05. Route with Agentic AI

In [ ]:
# Test the router with a question that belongs to AeroFlow.
# This demonstrates the main concept: the router selects the correct product-specific knowledge base
# before generating an answer, which is the core pattern behind agentic routing.
response = router_agent.invoke({"question": "What colors are available for AeroFlow?"})
print("\nResponse: ", response["answer"])

Selecting query engine: aeroflow

Response:  AeroFlow is available in **Coastal Blue, Sunset Orange, and Pearl White**.


In [ ]:
# Test the same workflow with a question for EcoSprint.
# This helps  see that the agent is not hard-coded to a single product and that the pattern
# scales to multiple specialized knowledge sources.
response = router_agent.invoke({"question": "What colors are available for EcoSprint?"})
print("\nResponse: ", response["answer"])

Selecting query engine: ecosprint

Response:  The EcoSprint is available in **Midnight Black, Ocean Blue, and Pearl White**.


: 